# Case study: what does a night in Berlin cost, and which differences are real?

*Session 5 · Author: course team · Licence: CC-BY-4.0*

Data: Inside Airbnb, Berlin snapshot of 26 June 2026 (CC BY 4.0), prepared with
`uv run python case-study/prepare_airbnb.py`. The listings were collected from public pages; host names and
texts have been removed. Report results in aggregate and never try to identify a host.

The notebook follows the three practice tasks of the session:

1. **Block 1**: describe prices, listings and demand; improve a poorly designed chart.
2. **Block 2**: compare groups (room type, district, superhost) with tests, confidence intervals and effect sizes;
   contingency tables; a simulated A/B test of a listing page.
3. **Block 3**: price against size (correlation, regression line); confounding of district comparisons; a
   one-page finding for a city housing analyst.

Session 4 showed that listings with a minimum stay of 28 nights or more report a price field that is not
comparable with short stays. Price analyses below therefore use the short-stay listings with a price.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.stats.contingency import association

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data" / "airbnb"
assert (DATA / "listings.parquet").exists(), "run: uv run python case-study/prepare_airbnb.py"

listings = pd.read_parquet(DATA / "listings.parquet")
monthly = pd.read_parquet(DATA / "reviews_monthly.parquet")
short = listings[listings["price"].notna() & listings["minimum_nights"].lt(28)].copy()
print(f"{len(listings):,} listings; {len(short):,} short-stay listings with a price")
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})

## Block 1 · Describing prices, listings and demand

### 1.1 The price distribution

**Exercise.** Before running the cell: guess the median price of a short-stay night in Berlin and the share of
listings above €300.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
sns.histplot(short, x="price", log_scale=True, bins=45, color="0.45", ax=left)
left.set(xlabel="EUR per night (log scale)", ylabel="listings", title="Short-stay prices")
order = ["Entire home/apt", "Hotel room", "Private room", "Shared room"]
sns.boxplot(short, x="price", y="room_type", order=order, log_scale=True, color="0.8", ax=right)
right.set(xlabel="EUR per night (log scale)", ylabel="", title="by room type")
plt.show()

print(short["price"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1))
print("share above 300 EUR:", round((short["price"] > 300).mean(), 3))
short.groupby("room_type")["price"].agg(["size", "median", "mean"]).round(1)

**Exercise.** Write two sentences that describe the price of a short-stay night for a reader without
statistics training: use the median and the 10th and 90th percentiles, not the mean and SD. Why is the mean
of the shared rooms a poor summary of only 80 listings?

### 1.2 Listings and prices by district

In [ ]:
by_district = (short.groupby("district")["price"].agg(listings="size", median="median")
               .join(listings["district"].value_counts().rename("all_listings"))
               .sort_values("median"))
by_district["share_entire_home"] = (short.assign(entire=short["room_type"].eq("Entire home/apt"))
                                    .groupby("district")["entire"].mean().round(3))
by_district

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2), layout="constrained")
ax.barh(by_district.index, by_district["median"], color="0.6")
ax.bar_label(ax.containers[0], labels=[f"€{v:.0f}" for v in by_district["median"]], padding=3, fontsize=8)
ax.set(xlabel="median price per short-stay night (EUR)", ylabel="",
       title=f"From €{by_district['median'].min():.0f} in Reinickendorf to €{by_district['median'].max():.0f} in Mitte")
plt.show()

### 1.3 Demand over time: reviews per month

A review is a **proxy** for a stay: not every guest writes one. The file also contains only listings that
still exist in 2026, so earlier years are understated (listings that left the platform are missing).

In [ ]:
per_month = monthly.groupby("month")["n_reviews"].sum()
fig, ax = plt.subplots(figsize=(9, 3.4), layout="constrained")
per_month.loc["2015":].plot(ax=ax, color="#0072B2")
ax.set(xlabel="", ylabel="reviews per month", title="Reviews per month of today's Berlin listings")
plt.show()
print(per_month.loc[["2019-07-01", "2020-04-01", "2025-07-01"]])

**Exercise.** Describe the pattern in three sentences: trend, season, break. In which month of the year are
reviews highest? (Hint: `per_month.groupby(per_month.index.month).mean()`.) Session 12 forecasts this series.

### 1.4 Improve a poorly designed chart

The cell draws the poor chart of the theory page. Improve it with the four critique questions (question,
encoding, accuracy, clarity): full axis from zero, one highlight colour for the pandemic years, a finding as
the title, and a caption with the limitation.

In [ ]:
yearly = monthly.groupby(monthly["month"].dt.year)["n_reviews"].sum().loc[2015:2025]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(yearly.index, yearly, color=plt.cm.rainbow(np.linspace(0, 1, len(yearly))))
ax.set_ylim(20000, 135000)
ax.set_title("Reviews")
ax.grid(True)
plt.show()

# your improved version here

## Block 2 · Comparing groups

### 2.1 Entire home vs private room, Mitte vs Neukölln, superhost vs not

In [ ]:
def compare(df, column, group, a_value, b_value):
    """Welch t-test, Mann-Whitney U and effect sizes for two groups of one column."""
    a = df.loc[df[group] == a_value, column].dropna()
    b = df.loc[df[group] == b_value, column].dropna()
    t = stats.ttest_ind(a, b, equal_var=False)
    ci = t.confidence_interval()
    mw = stats.mannwhitneyu(a, b)
    pooled_sd = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
    return pd.Series({"n_a": len(a), "n_b": len(b), "diff_of_means": a.mean() - b.mean(),
                      "ci_low": ci.low, "ci_high": ci.high, "p_t": t.pvalue, "p_mw": mw.pvalue,
                      "cohens_d": (a.mean() - b.mean()) / pooled_sd,
                      "cles": mw.statistic / (len(a) * len(b))})

short["log_price"] = np.log(short["price"])
results = pd.DataFrame({
    "entire vs private (EUR)": compare(short, "price", "room_type", "Entire home/apt", "Private room"),
    "entire vs private (log)": compare(short, "log_price", "room_type", "Entire home/apt", "Private room"),
    "Mitte vs Neukölln (EUR)": compare(short, "price", "district", "Mitte", "Neukölln"),
    "superhost vs not (EUR)": compare(short, "price", "host_is_superhost", True, False),
    "superhost vs not (rating)": compare(listings, "review_scores_rating", "host_is_superhost", True, False),
}).T
results.round(4)

**Exercises.**

1. For entire homes vs private rooms, Cohen's d is "medium" on the euro scale and large on the log scale, while
   the CLES is 0.84. Explain the disagreement in two sentences. Which effect size would you report?
2. For the superhost price comparison the t-test and the Mann–Whitney test disagree. Which question does each
   test answer? What would you tell a host who asks whether becoming a superhost lets them charge more?
3. Repeat the Mitte vs Neukölln comparison for entire homes only. Does the gap shrink?

### 2.2 Contingency tables

In [ ]:
multi = listings["calculated_host_listings_count"].gt(1).rename("multi_listing_host")
entry = listings["license"].notna().rename("licence_entry")
table = pd.crosstab(multi, entry)
res = stats.chi2_contingency(table)
print(table, "\n")
print(pd.crosstab(multi, entry, normalize="index").round(3))
print(f"chi2 = {res.statistic:.1f}, p = {res.pvalue:.1e}, V = {association(table, method='cramer'):.3f}")

In [ ]:
big = pd.crosstab(listings["district"], listings["room_type"])
res = stats.chi2_contingency(big)
print(f"chi2 = {res.statistic:.1f}, dof = {res.dof}, p = {res.pvalue:.1e}, "
      f"V = {association(big, method='cramer'):.3f}, cells with expected count < 5: {(res.expected_freq < 5).sum()}")
pd.crosstab(listings["district"], listings["room_type"], normalize="index").round(3)

**Exercise.** Write one sentence for each table for a city housing analyst: what differs, by how much, and
how strong is the association? Which of the two results is "significant but small"? What does the licence field
contain for multi-listing hosts (see Session 4, workbook 15), and why does that matter for your sentence?

### 2.3 A simulated A/B test of a listing page

A booking platform tests a new photo gallery on its listing pages. Primary metric: the share of visitors who
send a booking request (baseline 10 %). The simulation lets you see what the analysis does when you know the
truth.

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import confint_proportions_2indep, proportion_effectsize, proportions_ztest

n = int(np.ceil(NormalIndPower().solve_power(effect_size=proportion_effectsize(0.11, 0.10), alpha=0.05, power=0.8)))
print("planned users per group:", n)

rng = np.random.default_rng(7)
true_rate = {"A": 0.10, "B": 0.108}                  # the truth, unknown in a real test
users = {"A": rng.binomial(2 * n, 0.5)}              # random 50/50 assignment
users["B"] = 2 * n - users["A"]
conversions = {g: rng.binomial(users[g], true_rate[g]) for g in "AB"}
print(users, conversions)
print("sample-ratio check p =", round(stats.chisquare([users["A"], users["B"]]).pvalue, 3))
z, p = proportions_ztest([conversions["B"], conversions["A"]], [users["B"], users["A"]])
low, high = confint_proportions_2indep(conversions["B"], users["B"], conversions["A"], users["A"])
print(f"z = {z:.2f}, p = {p:.4f}, lift CI: {low:+.4f} to {high:+.4f}")

**Exercises.** (a) Re-run with different seeds. How often does the test detect the true lift of 0.8
points? Why not in 80 % of runs? (b) Simulate **peeking**: with no true effect (`true_rate` equal), check the
p-value after every 1,000 users and stop at the first p < 0.05. Over 200 simulated experiments, how often do you
declare a false win?

## Block 3 · Relationships, confounding and a finding

### 3.1 Price and size

In [ ]:
guests, price = short["accommodates"], short["price"]
print("Pearson raw:", round(stats.pearsonr(guests, price).statistic, 3))
print("Spearman:   ", round(stats.spearmanr(guests, price).statistic, 3))
print("Pearson log:", round(stats.pearsonr(guests, np.log(price)).statistic, 3))
keep = ~price.index.isin(price.nlargest(5).index)
print("Pearson raw without the 5 most expensive:", round(stats.pearsonr(guests[keep], price[keep]).statistic, 3))

b1, b0 = np.polyfit(guests, np.log(price), 1)
print(f"log price = {b0:.3f} + {b1:.3f} x guests -> +{np.exp(b1) - 1:.1%} per guest")
means = short.groupby("accommodates")["log_price"].agg(["mean", "size"]).query("size >= 30")
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.scatter(guests + np.random.default_rng(0).uniform(-0.25, 0.25, len(short)), np.log(price), s=3, alpha=0.1, color="0.5")
ax.plot(means.index, means["mean"], "o", color="#0072B2", label="mean per number of guests")
ax.plot([1, 12], [b0 + b1, b0 + 12 * b1], color="#D55E00", label="least-squares line")
ax.set(xlabel="guests", ylabel="log(price)", ylim=(np.log(15), np.log(1500)))
ax.legend(frameon=False)
plt.show()

**Exercise.** Is the rating related to the price? Compute Spearman's ρ between `review_scores_rating` and the
price, and between the rating and the price per guest. Interpret both in one sentence each.

### 3.2 Do district differences survive? Confounding by stay type, room type and size

In [ ]:
priced = listings[listings["price"].notna()].copy()
priced["stay"] = np.where(priced["minimum_nights"].ge(28), "medium-term", "short")
pair = priced[priced["district"].isin(["Friedrichshain-Kreuzberg", "Charlottenburg-Wilm."])]
print(pair.groupby("district")["price"].mean().round(1))
print(pair.groupby(["district", "stay"])["price"].agg(["size", "mean"]).round(1))

In [ ]:
import statsmodels.formula.api as smf

raw = smf.ols('log_price ~ C(district, Treatment("Mitte"))', data=short).fit()
adjusted = smf.ols('log_price ~ C(district, Treatment("Mitte")) + C(room_type) + accommodates', data=short).fit()
compare_districts = pd.DataFrame({
    "raw": np.exp(raw.params.filter(like="district")) - 1,
    "same room type and guests": np.exp(adjusted.params.filter(like="district")) - 1,
})
compare_districts.index = compare_districts.index.str.extract(r"T\.(.*)\]")[0]
compare_districts.sort_values("raw").round(3)

**Exercises.**

1. Explain in three sentences why Friedrichshain-Kreuzberg looks €10 more expensive than
   Charlottenburg-Wilmersdorf among all priced listings, but not within each type of stay. Is this a Simpson's
   paradox? Be precise about the size of the within-group differences.
2. Which districts' price gap to Mitte shrinks most when room type and guests are held fixed? Which variables
   might still confound the comparison?

### 3.3 A one-page finding

Write the four-part finding (finding first, one number with context, what it means for the reader, one
limitation) for a city housing analyst who asks: "Where in Berlin are short-term rentals most expensive, and
is that just because of the type of flat?" Use one chart. Alternatively, extend the dashboard
`dashboard_app.py` (exercises in its docstring) and start it with
`uv run streamlit run sessions/05-eda-and-statistics/workbooks/dashboard_app.py`.